#### Q1. First trace

Wrap the `rag()` method so each call produces a span. The simplest way
is to create a `RAGTraced` subclass of `RAGBase` that wraps `rag()`,
`search()`, and `llm()` each in their own span.

Run this query:

> How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary.
Count the spans in the console output - each one is a separate
`ReadableSpan` entry. How many spans does the trace produce?

* 1
* 3
* 5
* 7


In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

from starter import RAGBase, rag

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")


class RAGTraced(RAGBase):

    def rag(self, query: str) -> str:
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query: str):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt: str) -> str:
        with tracer.start_as_current_span("llm"):
            return super().llm(prompt)


rag_traced = RAGTraced(
    index=rag.index,
    model=rag.model,
    llm_client=rag.llm_client
)

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0xf7b9e99ef397286947bef4bf97cffede",
        "span_id": "0x8854f4ab939ba448",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xbe7ec86abfb2dda8",
    "start_time": "2026-07-22T17:05:44.803724Z",
    "end_time": "2026-07-22T17:05:44.806350Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "748ba193-0a88-41cc-b51d-12f33f7a8edc",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xf7b9e99ef397286947bef4bf97cffede",
        "span_id": "0x8ed3c3576be778c0",
        "trace_state": "[]"
    },
    "kind": "SpanKind

**Answer: 3 spans**

#### Q2. Capturing metrics as span attributes

Spans are not just timing markers - you can attach any information you
want to them with `set_attribute`. We already use spans to record how
long each step takes. Now we'll add the metrics we care about: tokens
and cost.

Read the token usage from the LLM response (the `llm()` method in the
starter already returns the raw response object) and set them as
attributes on the `llm` span:

```python
span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
```

And since we know both input and output tokens, we can also compute
the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

* 700
* 7000
* 70000
* 700000

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

from starter import RAGBase, rag

if not isinstance(trace.get_tracer_provider(), TracerProvider):
    provider = TracerProvider()
    provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
    trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")


class RAGTraced(RAGBase):

    def rag(self, query: str) -> str:
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query: str):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt: str) -> str:
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            if hasattr(response, "usage") and response.usage:
                usage = response.usage
                input_tokens = getattr(usage, "prompt_tokens",
                                       getattr(usage, "input_tokens", 0))
                output_tokens = getattr(
                    usage, "completion_tokens",
                    getattr(usage, "output_tokens", 0))

                # Registrar los atributos en el span
                span.set_attribute("input_tokens", input_tokens)
                span.set_attribute("output_tokens", output_tokens)

            return response


rag_traced = RAGTraced(index=rag.index,
                       model=rag.model,
                       llm_client=rag.llm_client)

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x18fef5a78fc69a4d19a3171df648e2fd",
        "span_id": "0x3ce077aaad14f4c7",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x0b7301c368ddbce0",
    "start_time": "2026-07-22T17:08:37.259583Z",
    "end_time": "2026-07-22T17:08:37.261109Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "748ba193-0a88-41cc-b51d-12f33f7a8edc",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x18fef5a78fc69a4d19a3171df648e2fd",
        "span_id": "0x2d7f2726406f6d28",
        "trace_state": "[]"
    },
    "kind": "SpanKind

For llm:  
    "start_time": 2026-07-22T17:08:37.261663Z"  
    "end_time": "2026-07-22T17:08:42.228400Z"  
    "input_tokens": 7111,  
    "output_tokens": 115  

**Answer: 7111 -> 7000**

#### Q3. Span timing

Each span automatically records its duration. Look at the console output
from Q1 and find the durations for the `search` span and the `llm` span.

For a typical query, roughly how long does the LLM call take?

* Under 100ms
* 100-500ms
* 500-2000ms
* Over 2000ms

> The first call can be slower (cold start). Pick the range you see
> most often.

llm    
    Start time: 2026-07-22T17:08:37.261663  
    End_time: 2026-07-22T17:08:42.228400  
    Duration: 4.966737 
    
**Answer: Over 2000ms**

#### Q4. Saving traces to SQLite

Re-run the query from Q1. Which span names appear in the `spans` table?

* Only `rag`
* `rag` and `llm`
* `rag`, `search`, and `llm`
* `search`, `llm`, and `judge`

In [4]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

from starter import RAGBase, rag

provider = TracerProvider()
exporter = SQLiteSpanExporter("traces.db")
provider.add_span_processor(SimpleSpanProcessor(exporter))

trace._TRACER_PROVIDER = None
trace.set_tracer_provider(provider)

tracer = provider.get_tracer("llm-zoomcamp")

class RAGTraced(RAGBase):

    def rag(self, query: str) -> str:
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query: str):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt: str) -> str:
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            if hasattr(response, "usage") and response.usage:
                usage = response.usage
                input_tokens = getattr(usage, "prompt_tokens",
                                       getattr(usage, "input_tokens", 0))
                output_tokens = getattr(
                    usage, "completion_tokens",
                    getattr(usage, "output_tokens", 0))

                # Registrar los atributos en el span
                span.set_attribute("input_tokens", input_tokens)
                span.set_attribute("output_tokens", output_tokens)

            return response

rag_traced = RAGTraced(
    index=rag.index, 
    model=rag.model, 
    llm_client=rag.llm_client
)

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

Overriding of current TracerProvider is not allowed


Answer: rag, search, and llm

#### Q5. Querying trace data

The traces are now in SQLite. Run one more query through the traced
RAG, then query the database.

The `rag` span wraps everything, so its duration includes both
`search` and `llm`. To see where time actually goes, exclude the
`rag` span and compare the children.

Using SQL (or pandas), compute the total duration for each span name
excluding `rag`. Which span type takes the most total time?

* `search`
* `llm`
* They're all about the same


In [13]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")

query = """
SELECT 
    name,
    COUNT(*) as total_calls,
    SUM(end_time - start_time) as total_duration_raw,
    AVG(end_time - start_time) as avg_duration_raw
FROM spans
WHERE name != 'rag'
GROUP BY name;
"""

df = pd.read_sql_query(query, conn)
print(df)

     name  total_calls  total_duration_raw  avg_duration_raw
0     llm            1          2422170104      2.422170e+09
1  search            1             1802292      1.802292e+06


**Answer: llm**

#### Q6. Token stability across runs

Load the SQLite data with pandas. One thing a dashboard can tell you
is how stable your system is. If the same query always produces the
same number of input tokens, the context your RAG retrieves is
consistent. If it varies a lot, something in the search may be
unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls
total in the database). Then compute the input tokens for each `llm`
span.

How much do the input tokens vary across these 4 runs?

* They're identical
* Within 10% of each other
* Within 50% of each other
* They vary more than 50%


In [19]:
for _ in range(4):
    rag_traced.rag(query)

In [20]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")

df_tokens = pd.read_sql_query("""
    SELECT 
        input_tokens 
    FROM spans 
    WHERE name = 'llm'
""", conn)

print("Tokens in each run:")
print(df_tokens)

Tokens in each run:
   input_tokens
0          6250
1          6250
2          6250
3          6250


**Answer**: They're identical